# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Define the concrete editorial interventions corresponding to the reason codes developed across w04_baseline_score.ipynb and w05_model.ipynb:high_demand_decay: Pages with $>1,000$ impressions and severe traffic decay ($<-50\%$).Editorial Action: Complete content overhaul. Rewrite outdated sections, re-evaluate search intent against top-ranking competitors, and restructure heading tags.stale_visible_page: Pages with high impressions, moderate drop, but age $>365$ days.Editorial Action: Fact & citation refresh. Update statistics, refresh publication timestamps, replace broken outbound links, and enhance introductory copy.moderate_decay_needs_refresh: Assets with drops between $-20\%$ and $-50\%$.Editorial Action: Metadata and on-page optimization. Refresh title tags, meta descriptions, internal anchor text, and add FAQs to capture secondary queries.

In [ ]:
import os
import numpy as np
import pandas as pd


df = pd.read_excel('capstone_data.xlsx')
df['trend_pct_num'] = pd.to_numeric(df['trend_pct'], errors='coerce').fillna(0)
df['impressions_90d'] = pd.to_numeric(
    df['impressions_90d'], errors='coerce'
).fillna(0)
df['clicks_90d'] = pd.to_numeric(df['clicks_90d'], errors='coerce').fillna(0)
df['content_age_days'] = pd.to_numeric(
    df['content_age_days'], errors='coerce'
).fillna(0)
df['feat_ctr_90d'] = np.where(
    df['impressions_90d'] > 0, df['clicks_90d'] / df['impressions_90d'], 0.0
)

# 2. Opportunity score calculation
df['opportunity_score'] = df['impressions_90d'] * (
    np.where(df['trend_pct_num'] < 0, np.abs(df['trend_pct_num']), 0) / 100.0
)


# 3. Rule / Reason Code Assignment
def assign_playbook_action(row):
  if (
      row['trend_direction'] != 'down'
      or row['impressions_90d'] < 1000
      or row['trend_pct_num'] > -20
  ):
    return 'IGNORE', 'no_action'
  elif row['trend_pct_num'] < -50:
    return 'FULL_REWRITE', 'high_demand_decay'
  elif row['content_age_days'] > 365:
    return 'REFRESH_METADATA_AND_FACTS', 'stale_visible_page'
  else:
    return 'OPTIMIZE_ON_PAGE', 'moderate_decay_needs_refresh'


df[['recommended_action', 'reason_code']] = df.apply(
    lambda r: pd.Series(assign_playbook_action(r)), axis=1
)

# 4. Filter and rank prioritized queue
playbook_queue = df[df['recommended_action'] != 'IGNORE'].sort_values(
    by='opportunity_score', ascending=False
)

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

import os
import numpy as np
import pandas as pd

# 1. Load data and compute clean historical features
df = pd.read_excel('capstone_data.xlsx')
df['trend_pct_num'] = pd.to_numeric(df['trend_pct'], errors='coerce').fillna(0)
df['impressions_90d'] = pd.to_numeric(
    df['impressions_90d'], errors='coerce'
).fillna(0)
df['clicks_90d'] = pd.to_numeric(df['clicks_90d'], errors='coerce').fillna(0)
df['content_age_days'] = pd.to_numeric(
    df['content_age_days'], errors='coerce'
).fillna(0)
df['feat_ctr_90d'] = np.where(
    df['impressions_90d'] > 0, df['clicks_90d'] / df['impressions_90d'], 0.0
)

# 2. Opportunity score calculation
df['opportunity_score'] = df['impressions_90d'] * (
    np.where(df['trend_pct_num'] < 0, np.abs(df['trend_pct_num']), 0) / 100.0
)


# 3. Rule / Reason Code Assignment
def assign_playbook_action(row):
  if (
      row['trend_direction'] != 'down'
      or row['impressions_90d'] < 1000
      or row['trend_pct_num'] > -20
  ):
    return 'IGNORE', 'no_action'
  elif row['trend_pct_num'] < -50:
    return 'FULL_REWRITE', 'high_demand_decay'
  elif row['content_age_days'] > 365:
    return 'REFRESH_METADATA_AND_FACTS', 'stale_visible_page'
  else:
    return 'OPTIMIZE_ON_PAGE', 'moderate_decay_needs_refresh'


df[['recommended_action', 'reason_code']] = df.apply(
    lambda r: pd.Series(assign_playbook_action(r)), axis=1
)

# 4. Filter and rank prioritized queue
playbook_queue = df[df['recommended_action'] != 'IGNORE'].sort_values(
    by='opportunity_score', ascending=False
)

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Explicitly list criteria where automation must not auto-assign updates without human sign-off:

Seasonal Content: Pages covering recurring annual events (e.g., Black Friday, holiday guides) that experience standard off-season drop velocity.

Canonical / Sunset URLs: Discontinued products or deprecated documentation that should be redirected rather than rewritten.

SERP Feature Displacements: Queries where Google introduced large interactive widgets or AI answers, reducing organic click potential regardless of content quality.

Legal / Compliance Articles: Regulated financial or healthcare advisories that require formal legal review before publishing changes

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Cost/Value Framework: Rewriting a high-priority URL costs an estimated 4–8 writer hours. Focusing updates strictly on the top 50 ranked pages (Precision@50: 94% on baseline, 70% on RF) ensures writer time is allocated exclusively to assets with sufficient search demand ($>1,000$ impressions).Retraining / Monitoring Triggers:Re-run monthly to absorb new rolling 90-day Search Console aggregations.Re-calibrate score weights if overall catalog drop rates shift by $>15\%$ following a major Google core algorithm update.Update feature inputs if CTR distribution shifts significantly due to layout changes.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# Export ranked queue for the research paper
os.makedirs('work/outputs', exist_ok=True)
export_cols = [
    'content_id',
    'opportunity_score',
    'recommended_action',
    'reason_code',
    'impressions_90d',
    'trend_pct_num',
    'content_age_days',
]
playbook_queue[export_cols].to_csv(
    'work/outputs/action_playbook_queue.csv', index=False
)
print(
    f'Exported {len(playbook_queue)} actionable items to'
    ' work/outputs/action_playbook_queue.csv'
)
playbook_queue[export_cols].head(10)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.